# Multivariate Time-Series Forecasting

## 1. Preprocessing

This notebook prepares the raw retail sales data for time-series forecasting.

The preprocessing pipeline will:

- Convert the sales data from wide format to a time-series-friendly structure.
- Connect daily sales with calendar information.
- Integrate weekly selling prices.
- Preserve product, store, department, category, and state information.
- Handle the high proportion of zero-sales observations appropriately.
- Prepare the data for feature engineering and forecasting models.

The raw CSV files will remain unchanged.

In [2]:
import os
import pandas as pd

# Raw dataset files
files = [
    "sales_train_validation.csv",
    "sales_train_evaluation.csv",
    "calendar.csv",
    "sell_prices.csv",
    "sample_submission.csv"
]

print("Checking raw dataset files...\n")

for file in files:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 ** 2)
        print(f"✓ {file:<35} {size_mb:>8.2f} MB")
    else:
        print(f"✗ {file:<35} NOT FOUND")

Checking raw dataset files...

✓ sales_train_validation.csv            114.45 MB
✓ sales_train_evaluation.csv            116.10 MB
✓ calendar.csv                            0.10 MB
✓ sell_prices.csv                       193.97 MB
✓ sample_submission.csv                   4.99 MB


In [3]:
import pandas as pd

sales_path = "sales_train_validation.csv"

# Read only the header
sales_columns = pd.read_csv(sales_path, nrows=0).columns.tolist()

print("Total columns:", len(sales_columns))
print("\nFirst 10 columns:")
print(sales_columns[:10])

print("\nLast 10 columns:")
print(sales_columns[-10:])

print("\nMetadata columns:")
print(sales_columns[:6])

print("\nNumber of daily sales columns:")
print(len(sales_columns) - 6)

Total columns: 1919

First 10 columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4']

Last 10 columns:
['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']

Metadata columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

Number of daily sales columns:
1913


In [4]:
import pandas as pd

sales_path = "sales_train_validation.csv"

# Only read metadata + first 10 sales days
sample_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
] + [f"d_{i}" for i in range(1, 11)]

sales_sample = pd.read_csv(
    sales_path,
    usecols=sample_cols,
    nrows=10
)

print("Shape:", sales_sample.shape)

print("\nData:")
display(sales_sample)

print("\nData types:")
print(sales_sample.dtypes)

print("\nMissing values:")
print(sales_sample.isna().sum())

Shape: (10, 16)

Data:


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,12,15,0,0,0,4,6,5,7,0
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,2,0,7,3,0,2,3,9,0,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,0,0,0,0,0,0,0



Data types:
id          object
item_id     object
dept_id     object
cat_id      object
store_id    object
state_id    object
d_1          int64
d_2          int64
d_3          int64
d_4          int64
d_5          int64
d_6          int64
d_7          int64
d_8          int64
d_9          int64
d_10         int64
dtype: object

Missing values:
id          0
item_id     0
dept_id     0
cat_id      0
store_id    0
state_id    0
d_1         0
d_2         0
d_3         0
d_4         0
d_5         0
d_6         0
d_7         0
d_8         0
d_9         0
d_10        0
dtype: int64


In [5]:
import pandas as pd

calendar_path = "calendar.csv"

calendar = pd.read_csv(calendar_path)

print("Calendar shape:", calendar.shape)

print("\nColumns:")
print(calendar.columns.tolist())

print("\nFirst 5 rows:")
display(calendar.head())

print("\nLast 5 rows:")
display(calendar.tail())

print("\nMissing values:")
print(calendar.isna().sum())

Calendar shape: (1969, 14)

Columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

First 5 rows:


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1



Last 5 rows:


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
1964,2016-06-15,11620,Wednesday,5,6,2016,d_1965,NaN,NaN,NaN,NaN,0,1,1
1965,2016-06-16,11620,Thursday,6,6,2016,d_1966,NaN,NaN,NaN,NaN,0,0,0
1966,2016-06-17,11620,Friday,7,6,2016,d_1967,NaN,NaN,NaN,NaN,0,0,0
1967,2016-06-18,11621,Saturday,1,6,2016,d_1968,NaN,NaN,NaN,NaN,0,0,0
1968,2016-06-19,11621,Sunday,2,6,2016,d_1969,NBAFinalsEnd,Sporting,Father's day,Cultural,0,0,0



Missing values:
date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64


In [6]:
# Convert date column to datetime
calendar["date"] = pd.to_datetime(calendar["date"])

# Check uniqueness
print("Unique d values:", calendar["d"].nunique())
print("Total calendar rows:", len(calendar))

print("\nUnique dates:", calendar["date"].nunique())

# Check expected sales range
sales_days = [f"d_{i}" for i in range(1, 1914)]

missing_days = set(sales_days) - set(calendar["d"])
extra_days = set(calendar["d"]) - set(sales_days)

print("\nSales days expected:", len(sales_days))
print("Missing from calendar:", len(missing_days))
print("Extra calendar days:", len(extra_days))

# Show mapping
print("\nFirst 5 mappings:")
display(calendar[["d", "date"]].head())

print("\nLast 5 mappings:")
display(calendar[["d", "date"]].tail())

Unique d values: 1969
Total calendar rows: 1969

Unique dates: 1969

Sales days expected: 1913
Missing from calendar: 0
Extra calendar days: 56

First 5 mappings:


,d,date
0,d_1,2011-01-29
1,d_2,2011-01-30
2,d_3,2011-01-31
3,d_4,2011-02-01
4,d_5,2011-02-02



Last 5 mappings:


,d,date
1964,d_1965,2016-06-15
1965,d_1966,2016-06-16
1966,d_1967,2016-06-17
1967,d_1968,2016-06-18
1968,d_1969,2016-06-19


In [8]:
# Keep only the historical training period
calendar_train = calendar[
    calendar["d"].isin(sales_days)
].copy()

# Extract numeric day number from d_1, d_2, ..., d_1913
calendar_train["day_num"] = (
    calendar_train["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

# Sort chronologically
calendar_train = (
    calendar_train
    .sort_values("day_num")
    .reset_index(drop=True)
)

print("Training calendar shape:", calendar_train.shape)

print("\nDate range:")
print("Start:", calendar_train["date"].min())
print("End  :", calendar_train["date"].max())

print("\nUnique days:", calendar_train["d"].nunique())
print("Unique dates:", calendar_train["date"].nunique())

print("\nFirst 5 days:")
display(
    calendar_train[
        ["d", "day_num", "date", "wm_yr_wk", "weekday", "month", "year"]
    ].head()
)

print("\nLast 5 days:")
display(
    calendar_train[
        ["d", "day_num", "date", "wm_yr_wk", "weekday", "month", "year"]
    ].tail()
)

print(
    "\nDay numbers continuous:",
    calendar_train["day_num"].equals(
        pd.Series(range(1, 1914))
    )
)

Training calendar shape: (1913, 15)

Date range:
Start: 2011-01-29 00:00:00
End  : 2016-04-24 00:00:00

Unique days: 1913
Unique dates: 1913

First 5 days:


,d,day_num,date,wm_yr_wk,weekday,month,year
0,d_1,1,2011-01-29,11101,Saturday,1,2011
1,d_2,2,2011-01-30,11101,Sunday,1,2011
2,d_3,3,2011-01-31,11101,Monday,1,2011
3,d_4,4,2011-02-01,11101,Tuesday,2,2011
4,d_5,5,2011-02-02,11101,Wednesday,2,2011



Last 5 days:


,d,day_num,date,wm_yr_wk,weekday,month,year
1908,d_1909,1909,2016-04-20,11612,Wednesday,4,2016
1909,d_1910,1910,2016-04-21,11612,Thursday,4,2016
1910,d_1911,1911,2016-04-22,11612,Friday,4,2016
1911,d_1912,1912,2016-04-23,11613,Saturday,4,2016
1912,d_1913,1913,2016-04-24,11613,Sunday,4,2016



Day numbers continuous: True


In [9]:
import pandas as pd

sales_path = "sales_train_validation.csv"

# Metadata columns
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Daily sales columns
day_cols = [f"d_{i}" for i in range(1, 1914)]

# Read only one chunk for testing
sales_chunk = pd.read_csv(
    sales_path,
    usecols=id_cols + day_cols,
    nrows=1000
)

print("Original chunk shape:", sales_chunk.shape)

# Convert wide → long
sales_long_test = sales_chunk.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

print("\nLong-format test shape:", sales_long_test.shape)

print("\nFirst 5 rows:")
display(sales_long_test.head())

print("\nData types:")
print(sales_long_test.dtypes)

print("\nMissing sales:", sales_long_test["sales"].isna().sum())

print("\nUnique products:", sales_long_test["item_id"].nunique())
print("Unique stores:", sales_long_test["store_id"].nunique())
print("Unique days:", sales_long_test["d"].nunique())

Original chunk shape: (1000, 1919)

Long-format test shape: (1913000, 8)

First 5 rows:


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0



Data types:
id          object
item_id     object
dept_id     object
cat_id      object
store_id    object
state_id    object
d           object
sales        int64
dtype: object

Missing sales: 0

Unique products: 1000
Unique stores: 1
Unique days: 1913


In [11]:
import os
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

sales_path = "sales_train_validation.csv"
output_path = "sales_train_long.parquet"

# Columns
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

day_cols = [f"d_{i}" for i in range(1, 1914)]

# Remove incomplete output if it already exists
if os.path.exists(output_path):
    os.remove(output_path)
    print("Removed incomplete Parquet file.")

chunk_size = 1000
total_rows = 0
writer = None

for chunk_num, sales_chunk in enumerate(
    pd.read_csv(
        sales_path,
        usecols=id_cols + day_cols,
        chunksize=chunk_size
    ),
    start=1
):

    # Wide → Long
    sales_long = sales_chunk.melt(
        id_vars=id_cols,
        value_vars=day_cols,
        var_name="d",
        value_name="sales"
    )

    # Convert pandas DataFrame to Arrow table
    table = pa.Table.from_pandas(
        sales_long,
        preserve_index=False
    )

    # Create writer using first chunk schema
    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    # Write current chunk
    writer.write_table(table)

    total_rows += len(sales_long)

    print(
        f"Chunk {chunk_num}: "
        f"{len(sales_chunk):,} products → "
        f"{len(sales_long):,} observations"
    )

    del sales_chunk, sales_long, table
    gc.collect()

# Close Parquet writer
if writer is not None:
    writer.close()

print("\nConversion completed.")
print("Total observations:", f"{total_rows:,}")

# Check output file
file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

print("Output file:", output_path)
print("File size:", f"{file_size_mb:.2f} MB")

Removed incomplete Parquet file.
Chunk 1: 1,000 products → 1,913,000 observations
Chunk 2: 1,000 products → 1,913,000 observations
Chunk 3: 1,000 products → 1,913,000 observations
Chunk 4: 1,000 products → 1,913,000 observations
Chunk 5: 1,000 products → 1,913,000 observations
Chunk 6: 1,000 products → 1,913,000 observations
Chunk 7: 1,000 products → 1,913,000 observations
Chunk 8: 1,000 products → 1,913,000 observations
Chunk 9: 1,000 products → 1,913,000 observations
Chunk 10: 1,000 products → 1,913,000 observations
Chunk 11: 1,000 products → 1,913,000 observations
Chunk 12: 1,000 products → 1,913,000 observations
Chunk 13: 1,000 products → 1,913,000 observations
Chunk 14: 1,000 products → 1,913,000 observations
Chunk 15: 1,000 products → 1,913,000 observations
Chunk 16: 1,000 products → 1,913,000 observations
Chunk 17: 1,000 products → 1,913,000 observations
Chunk 18: 1,000 products → 1,913,000 observations
Chunk 19: 1,000 products → 1,913,000 observations
Chunk 20: 1,000 products →

In [12]:
import pyarrow.parquet as pq

output_path = "sales_train_long.parquet"

# Read Parquet metadata only
parquet_file = pq.ParquetFile(output_path)

print("Number of rows:", f"{parquet_file.metadata.num_rows:,}")
print("Number of columns:", parquet_file.metadata.num_columns)

print("\nColumns:")
print(parquet_file.schema.names)

print("\nNumber of row groups:", parquet_file.num_row_groups)

Number of rows: 58,327,370
Number of columns: 8

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales']

Number of row groups: 61


In [13]:
# Read only a small sample from the Parquet file
sample = pd.read_parquet(
    output_path,
    columns=[
        "id",
        "item_id",
        "store_id",
        "state_id",
        "d",
        "sales"
    ]
).head(10)

print("\nSample:")
display(sample)


Sample:


,id,item_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,CA_1,CA,d_1,0
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,CA_1,CA,d_1,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,CA_1,CA,d_1,0
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,CA_1,CA,d_1,12
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,CA_1,CA,d_1,2
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,CA_1,CA,d_1,0


In [14]:
# Calendar columns needed for merging
calendar_features = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

calendar_lookup = calendar_train[calendar_features].copy()

# Make sure the join key is unique
print("Calendar rows:", len(calendar_lookup))
print("Unique d:", calendar_lookup["d"].nunique())

print(
    "\nDuplicate d values:",
    calendar_lookup["d"].duplicated().sum()
)

print("\nCalendar lookup sample:")
display(calendar_lookup.head())

Calendar rows: 1913
Unique d: 1913

Duplicate d values: 0

Calendar lookup sample:


,d,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,d_1,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,d_2,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,d_3,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,d_4,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,d_5,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,0,1


In [15]:
import os
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

sales_long_path = "sales_train_long.parquet"
output_path = "sales_calendar_train.parquet"

# Remove output if it already exists
if os.path.exists(output_path):
    os.remove(output_path)
    print("Removed existing output file.")

# Calendar lookup
calendar_merge = calendar_lookup.copy()

# Make sure date is datetime
calendar_merge["date"] = pd.to_datetime(calendar_merge["date"])

# Open existing sales Parquet
sales_parquet = pq.ParquetFile(sales_long_path)

writer = None
total_rows = 0

print("Processing row groups...\n")

for group_num in range(sales_parquet.num_row_groups):

    # Read one row group
    sales_group = sales_parquet.read_row_group(group_num).to_pandas()

    # Merge sales with calendar
    merged_group = sales_group.merge(
        calendar_merge,
        on="d",
        how="left",
        validate="many_to_one"
    )

    # Convert to Arrow
    table = pa.Table.from_pandas(
        merged_group,
        preserve_index=False
    )

    # Create writer using first group
    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    # Write merged group
    writer.write_table(table)

    total_rows += len(merged_group)

    print(
        f"Row group {group_num + 1}/{sales_parquet.num_row_groups}: "
        f"{len(merged_group):,} rows"
    )

    del sales_group, merged_group, table
    gc.collect()

# Close writer
if writer is not None:
    writer.close()

print("\nMerge completed.")
print("Total rows:", f"{total_rows:,}")

file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

print("Output file:", output_path)
print("File size:", f"{file_size_mb:.2f} MB")

Processing row groups...

Row group 1/61: 1,048,576 rows
Row group 2/61: 864,424 rows
Row group 3/61: 1,048,576 rows
Row group 4/61: 864,424 rows
Row group 5/61: 1,048,576 rows
Row group 6/61: 864,424 rows
Row group 7/61: 1,048,576 rows
Row group 8/61: 864,424 rows
Row group 9/61: 1,048,576 rows
Row group 10/61: 864,424 rows
Row group 11/61: 1,048,576 rows
Row group 12/61: 864,424 rows
Row group 13/61: 1,048,576 rows
Row group 14/61: 864,424 rows
Row group 15/61: 1,048,576 rows
Row group 16/61: 864,424 rows
Row group 17/61: 1,048,576 rows
Row group 18/61: 864,424 rows
Row group 19/61: 1,048,576 rows
Row group 20/61: 864,424 rows
Row group 21/61: 1,048,576 rows
Row group 22/61: 864,424 rows
Row group 23/61: 1,048,576 rows
Row group 24/61: 864,424 rows
Row group 25/61: 1,048,576 rows
Row group 26/61: 864,424 rows
Row group 27/61: 1,048,576 rows
Row group 28/61: 864,424 rows
Row group 29/61: 1,048,576 rows
Row group 30/61: 864,424 rows
Row group 31/61: 1,048,576 rows
Row group 32/61: 864,

In [16]:
import pyarrow.parquet as pq
import pandas as pd

merged_path = "sales_calendar_train.parquet"

merged_parquet = pq.ParquetFile(merged_path)

print("Number of rows:",
      f"{merged_parquet.metadata.num_rows:,}")

print("Number of columns:",
      merged_parquet.metadata.num_columns)

print("\nColumns:")
print(merged_parquet.schema.names)

print("\nNumber of row groups:",
      merged_parquet.num_row_groups)

Number of rows: 58,327,370
Number of columns: 21

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Number of row groups: 61


In [17]:
# Read only the first row group
sample_group = merged_parquet.read_row_group(0).to_pandas()

print("Sample row-group shape:", sample_group.shape)

print("\nSample:")
display(
    sample_group[
        [
            "id",
            "item_id",
            "store_id",
            "state_id",
            "d",
            "sales",
            "date",
            "wm_yr_wk",
            "weekday",
            "month",
            "year",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ].head(10)
)

print("\nMissing values in calendar fields:")

calendar_check_cols = [
    "date",
    "wm_yr_wk",
    "weekday",
    "month",
    "year",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

print(sample_group[calendar_check_cols].isna().sum())

Sample row-group shape: (1048576, 21)

Sample:


,id,item_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,month,year,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,CA_1,CA,d_1,12,2011-01-29,11101,Saturday,1,2011,0,0,0
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,CA_1,CA,d_1,2,2011-01-29,11101,Saturday,1,2011,0,0,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,2011,0,0,0



Missing values in calendar fields:
date        0
wm_yr_wk    0
weekday     0
month       0
year        0
snap_CA     0
snap_TX     0
snap_WI     0
dtype: int64


In [18]:
import pandas as pd

price_path = "sell_prices.csv"

price_sample = pd.read_csv(
    price_path,
    nrows=10
)

print("Sample shape:", price_sample.shape)

print("\nColumns:")
print(price_sample.columns.tolist())

print("\nSample:")
display(price_sample)

print("\nData types:")
print(price_sample.dtypes)

print("\nMissing values:")
print(price_sample.isna().sum())

Sample shape: (10, 4)

Columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Sample:


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
5,CA_1,HOBBIES_1_001,11330,8.26
6,CA_1,HOBBIES_1_001,11331,8.26
7,CA_1,HOBBIES_1_001,11332,8.26
8,CA_1,HOBBIES_1_001,11333,8.26
9,CA_1,HOBBIES_1_001,11334,8.26



Data types:
store_id       object
item_id        object
wm_yr_wk        int64
sell_price    float64
dtype: object

Missing values:
store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64


In [19]:
import pandas as pd

price_path = "sell_prices.csv"

# Read only the columns needed for validation
price_keys = pd.read_csv(
    price_path,
    usecols=["store_id", "item_id", "wm_yr_wk"]
)

print("Price rows:", f"{len(price_keys):,}")

print("\nUnique stores:", price_keys["store_id"].nunique())
print("Unique products:", price_keys["item_id"].nunique())
print("Unique weeks:", price_keys["wm_yr_wk"].nunique())

# Check duplicate join keys
duplicate_keys = price_keys.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print("\nDuplicate (store_id, item_id, wm_yr_wk) keys:", duplicate_keys)

# Number of unique combinations
unique_keys = price_keys[
    ["store_id", "item_id", "wm_yr_wk"]
].drop_duplicates()

print("Unique join keys:", f"{len(unique_keys):,}")

Price rows: 6,841,121

Unique stores: 10
Unique products: 3049
Unique weeks: 282

Duplicate (store_id, item_id, wm_yr_wk) keys: 0
Unique join keys: 6,841,121


In [20]:
import os
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

price_path = "sell_prices.csv"
price_output_path = "sell_prices.parquet"

# Remove previous output if it exists
if os.path.exists(price_output_path):
    os.remove(price_output_path)

# Free validation dataframe
del price_keys
gc.collect()

writer = None
total_rows = 0

for chunk_num, price_chunk in enumerate(
    pd.read_csv(
        price_path,
        usecols=[
            "store_id",
            "item_id",
            "wm_yr_wk",
            "sell_price"
        ],
        chunksize=500_000
    ),
    start=1
):

    table = pa.Table.from_pandas(
        price_chunk,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            price_output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    total_rows += len(price_chunk)

    print(
        f"Chunk {chunk_num}: "
        f"{len(price_chunk):,} rows"
    )

    del price_chunk, table
    gc.collect()

if writer is not None:
    writer.close()

print("\nPrice conversion completed.")
print("Total rows:", f"{total_rows:,}")

file_size_mb = os.path.getsize(price_output_path) / (1024 ** 2)

print("Output file:", price_output_path)
print("File size:", f"{file_size_mb:.2f} MB")

Chunk 1: 500,000 rows
Chunk 2: 500,000 rows
Chunk 3: 500,000 rows
Chunk 4: 500,000 rows
Chunk 5: 500,000 rows
Chunk 6: 500,000 rows
Chunk 7: 500,000 rows
Chunk 8: 500,000 rows
Chunk 9: 500,000 rows
Chunk 10: 500,000 rows
Chunk 11: 500,000 rows
Chunk 12: 500,000 rows
Chunk 13: 500,000 rows
Chunk 14: 341,121 rows

Price conversion completed.
Total rows: 6,841,121
Output file: sell_prices.parquet
File size: 1.63 MB


In [21]:
import pyarrow.parquet as pq
import pandas as pd

sales_calendar_path = "sales_calendar_train.parquet"
price_path = "sell_prices.parquet"

# Read first sales row group only
sales_test = pq.ParquetFile(
    sales_calendar_path
).read_row_group(0).to_pandas()

# Read price data
price_test = pd.read_parquet(price_path)

print("Sales test rows:", f"{len(sales_test):,}")
print("Price rows:", f"{len(price_test):,}")

# Merge on product + store + week
merged_test = sales_test.merge(
    price_test,
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

print("\nMerged rows:", f"{len(merged_test):,}")

print(
    "Missing sell_price:",
    merged_test["sell_price"].isna().sum()
)

print(
    "Missing sell_price %:",
    f"{merged_test['sell_price'].isna().mean() * 100:.4f}%"
)

print("\nSample:")
display(
    merged_test[
        [
            "id",
            "item_id",
            "store_id",
            "d",
            "sales",
            "date",
            "wm_yr_wk",
            "sell_price"
        ]
    ].head(10)
)

Sales test rows: 1,048,576
Price rows: 6,841,121

Merged rows: 1,048,576
Missing sell_price: 367871
Missing sell_price %: 35.0829%

Sample:


,id,item_id,store_id,d,sales,date,wm_yr_wk,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_1,0,2011-01-29,11101,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,CA_1,d_1,0,2011-01-29,11101,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,CA_1,d_1,0,2011-01-29,11101,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,CA_1,d_1,0,2011-01-29,11101,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,CA_1,d_1,0,2011-01-29,11101,NaN
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,CA_1,d_1,0,2011-01-29,11101,NaN
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,CA_1,d_1,0,2011-01-29,11101,NaN
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,CA_1,d_1,12,2011-01-29,11101,0.46
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,CA_1,d_1,2,2011-01-29,11101,1.56
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,CA_1,d_1,0,2011-01-29,11101,3.17


In [22]:
import gc
import pandas as pd
import pyarrow.parquet as pq

sales_calendar_path = "sales_calendar_train.parquet"
price_path = "sell_prices.parquet"

# Free previous test data
del sales_test
del price_test
del merged_test
gc.collect()

sales_parquet = pq.ParquetFile(sales_calendar_path)
price_parquet = pq.ParquetFile(price_path)

# Load price data
# Only 4 columns are needed
price_df = price_parquet.read().to_pandas()

print("Price rows loaded:", f"{len(price_df):,}")

# Create lookup key
price_df["price_key"] = (
    price_df["item_id"].astype(str)
    + "_"
    + price_df["store_id"].astype(str)
    + "_"
    + price_df["wm_yr_wk"].astype(str)
)

price_keys = set(price_df["price_key"])

print("Unique price keys:", f"{len(price_keys):,}")

# Check sales coverage row-group by row-group
total_sales_rows = 0
matched_rows = 0

for group_num in range(sales_parquet.num_row_groups):

    sales_group = sales_parquet.read_row_group(group_num).to_pandas()

    sales_group["price_key"] = (
        sales_group["item_id"].astype(str)
        + "_"
        + sales_group["store_id"].astype(str)
        + "_"
        + sales_group["wm_yr_wk"].astype(str)
    )

    matched = sales_group["price_key"].isin(price_keys)

    total_sales_rows += len(sales_group)
    matched_rows += matched.sum()

    print(
        f"Row group {group_num + 1}/{sales_parquet.num_row_groups} "
        f"→ matched: {matched.sum():,} / {len(sales_group):,}"
    )

    del sales_group, matched
    gc.collect()

missing_rows = total_sales_rows - matched_rows

print("\n====================================")
print("Price coverage summary")
print("====================================")

print("Total sales rows:", f"{total_sales_rows:,}")
print("Matched price rows:", f"{matched_rows:,}")
print("Missing price rows:", f"{missing_rows:,}")

print(
    "Missing price %:",
    f"{missing_rows / total_sales_rows * 100:.2f}%"
)

print(
    "Price coverage %:",
    f"{matched_rows / total_sales_rows * 100:.2f}%"
)

Price rows loaded: 6,841,121
Unique price keys: 6,841,121
Row group 1/61 → matched: 680,705 / 1,048,576
Row group 2/61 → matched: 839,791 / 864,424
Row group 3/61 → matched: 737,209 / 1,048,576
Row group 4/61 → matched: 839,763 / 864,424
Row group 5/61 → matched: 698,841 / 1,048,576
Row group 6/61 → matched: 831,077 / 864,424
Row group 7/61 → matched: 670,806 / 1,048,576
Row group 8/61 → matched: 836,278 / 864,424
Row group 9/61 → matched: 667,902 / 1,048,576
Row group 10/61 → matched: 795,348 / 864,424
Row group 11/61 → matched: 506,854 / 1,048,576
Row group 12/61 → matched: 728,182 / 864,424
Row group 13/61 → matched: 662,590 / 1,048,576
Row group 14/61 → matched: 832,958 / 864,424
Row group 15/61 → matched: 731,146 / 1,048,576
Row group 16/61 → matched: 840,772 / 864,424
Row group 17/61 → matched: 695,521 / 1,048,576
Row group 18/61 → matched: 832,458 / 864,424
Row group 19/61 → matched: 630,245 / 1,048,576
Row group 20/61 → matched: 827,006 / 864,424
Row group 21/61 → matched: 698,

In [23]:
import gc
import pandas as pd
import pyarrow.parquet as pq

sales_calendar_path = "sales_calendar_train.parquet"

sales_parquet = pq.ParquetFile(sales_calendar_path)

# Statistics
total_rows = 0
price_available_rows = 0
price_missing_rows = 0

sales_available_price = 0
sales_missing_price = 0

zero_sales_available_price = 0
zero_sales_missing_price = 0

for group_num in range(sales_parquet.num_row_groups):

    sales_group = sales_parquet.read_row_group(
        group_num,
        columns=[
            "item_id",
            "store_id",
            "wm_yr_wk",
            "sales"
        ]
    ).to_pandas()

    # Merge only the price column
    group = sales_group.merge(
        price_df,
        on=["item_id", "store_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one"
    )

    price_available = group["sell_price"].notna()
    price_missing = group["sell_price"].isna()

    total_rows += len(group)

    price_available_rows += price_available.sum()
    price_missing_rows += price_missing.sum()

    sales_available_price += group.loc[
        price_available, "sales"
    ].sum()

    sales_missing_price += group.loc[
        price_missing, "sales"
    ].sum()

    zero_sales_available_price += (
        price_available & (group["sales"] == 0)
    ).sum()

    zero_sales_missing_price += (
        price_missing & (group["sales"] == 0)
    ).sum()

    print(
        f"Row group {group_num + 1}/{sales_parquet.num_row_groups} processed"
    )

    del sales_group, group
    gc.collect()


print("\n====================================")
print("Missing Price Analysis")
print("====================================")

print("\nTotal observations:",
      f"{total_rows:,}")

print("\nPrice available:",
      f"{price_available_rows:,}",
      f"({price_available_rows / total_rows * 100:.2f}%)")

print("Price missing:",
      f"{price_missing_rows:,}",
      f"({price_missing_rows / total_rows * 100:.2f}%)")

print("\nAverage sales when price is available:")
print(
    sales_available_price / price_available_rows
)

print("\nAverage sales when price is missing:")
print(
    sales_missing_price / price_missing_rows
)

print("\nZero-sales % when price is available:")
print(
    zero_sales_available_price /
    price_available_rows * 100
)

print("\nZero-sales % when price is missing:")
print(
    zero_sales_missing_price /
    price_missing_rows * 100
)

Row group 1/61 processed
Row group 2/61 processed
Row group 3/61 processed
Row group 4/61 processed
Row group 5/61 processed
Row group 6/61 processed
Row group 7/61 processed
Row group 8/61 processed
Row group 9/61 processed
Row group 10/61 processed
Row group 11/61 processed
Row group 12/61 processed
Row group 13/61 processed
Row group 14/61 processed
Row group 15/61 processed
Row group 16/61 processed
Row group 17/61 processed
Row group 18/61 processed
Row group 19/61 processed
Row group 20/61 processed
Row group 21/61 processed
Row group 22/61 processed
Row group 23/61 processed
Row group 24/61 processed
Row group 25/61 processed
Row group 26/61 processed
Row group 27/61 processed
Row group 28/61 processed
Row group 29/61 processed
Row group 30/61 processed
Row group 31/61 processed
Row group 32/61 processed
Row group 33/61 processed
Row group 34/61 processed
Row group 35/61 processed
Row group 36/61 processed
Row group 37/61 processed
Row group 38/61 processed
Row group 39/61 proce

In [24]:
import gc
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

sales_calendar_path = "sales_calendar_train.parquet"
output_path = "sales_calendar_price_train.parquet"

sales_parquet = pq.ParquetFile(sales_calendar_path)

writer = None
total_rows = 0

for group_num in range(sales_parquet.num_row_groups):

    sales_group = sales_parquet.read_row_group(group_num).to_pandas()

    # Merge selling price
    merged_group = sales_group.merge(
        price_df,
        on=["item_id", "store_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one"
    )

    # Price availability indicator
    merged_group["price_available"] = (
        merged_group["sell_price"].notna().astype("int8")
    )

    total_rows += len(merged_group)

    # Convert to Arrow
    table = pa.Table.from_pandas(
        merged_group,
        preserve_index=False
    )

    # Create writer only once
    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    print(
        f"Row group {group_num + 1}/{sales_parquet.num_row_groups} "
        f"processed"
    )

    del sales_group, merged_group, table
    gc.collect()

writer.close()

print("\n====================================")
print("Sales + Calendar + Price Merge")
print("====================================")

print("Total rows:", f"{total_rows:,}")
print("Output file:", output_path)

Row group 1/61 processed
Row group 2/61 processed
Row group 3/61 processed
Row group 4/61 processed
Row group 5/61 processed
Row group 6/61 processed
Row group 7/61 processed
Row group 8/61 processed
Row group 9/61 processed
Row group 10/61 processed
Row group 11/61 processed
Row group 12/61 processed
Row group 13/61 processed
Row group 14/61 processed
Row group 15/61 processed
Row group 16/61 processed
Row group 17/61 processed
Row group 18/61 processed
Row group 19/61 processed
Row group 20/61 processed
Row group 21/61 processed
Row group 22/61 processed
Row group 23/61 processed
Row group 24/61 processed
Row group 25/61 processed
Row group 26/61 processed
Row group 27/61 processed
Row group 28/61 processed
Row group 29/61 processed
Row group 30/61 processed
Row group 31/61 processed
Row group 32/61 processed
Row group 33/61 processed
Row group 34/61 processed
Row group 35/61 processed
Row group 36/61 processed
Row group 37/61 processed
Row group 38/61 processed
Row group 39/61 proce

In [25]:
import pyarrow.parquet as pq

output_path = "sales_calendar_price_train.parquet"

parquet_file = pq.ParquetFile(output_path)

print("====================================")
print("Final Preprocessed Dataset")
print("====================================")

print("Rows:", f"{parquet_file.metadata.num_rows:,}")
print("Columns:", parquet_file.schema.names)
print("Number of row groups:", parquet_file.num_row_groups)

# Read only first row group for verification
sample = parquet_file.read_row_group(0).to_pandas()

print("\nSample shape:", sample.shape)

print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 rows:")
print(sample.head())

print("\nPrice availability:")
print(sample["price_available"].value_counts())

print("\nPrice missing:")
print(
    sample["sell_price"].isna().sum()
)

print("\nSales missing:")
print(
    sample["sales"].isna().sum()
)

print("\n====================================")
print("Verification")
print("====================================")

print(
    "Row count preserved:",
    parquet_file.metadata.num_rows == 58_327_370
)

print(
    "Expected price column present:",
    "sell_price" in parquet_file.schema.names
)

print(
    "Price indicator present:",
    "price_available" in parquet_file.schema.names
)

Final Preprocessed Dataset
Rows: 58,327,370
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_key', 'price_available']
Number of row groups: 61

Sample shape: (1048576, 24)

Data types:
id                         object
item_id                    object
dept_id                    object
cat_id                     object
store_id                   object
state_id                   object
d                          object
sales                       int64
date               datetime64[ns]
wm_yr_wk                    int64
weekday                    object
wday                        int64
month                       int64
year                        int64
event_name_1               object
event_type_1               object
event_name_2               object
event_type_2              

In [26]:
import pyarrow.parquet as pq
import pyarrow as pa
import gc
import os

input_path = "sales_calendar_price_train.parquet"
output_path = "preprocessed_train.parquet"

parquet_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0

for group_num in range(parquet_file.num_row_groups):

    df = parquet_file.read_row_group(group_num).to_pandas()

    # Remove helper column created only for price matching
    if "price_key" in df.columns:
        df = df.drop(columns=["price_key"])

    total_rows += len(df)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    print(
        f"Row group {group_num + 1}/{parquet_file.num_row_groups} processed"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Preprocessing Cleanup Complete")
print("====================================")

print("Total rows:", f"{total_rows:,}")
print("Output file:", output_path)
print("File size (MB):", round(os.path.getsize(output_path) / (1024**2), 2))

Row group 1/61 processed
Row group 2/61 processed
Row group 3/61 processed
Row group 4/61 processed
Row group 5/61 processed
Row group 6/61 processed
Row group 7/61 processed
Row group 8/61 processed
Row group 9/61 processed
Row group 10/61 processed
Row group 11/61 processed
Row group 12/61 processed
Row group 13/61 processed
Row group 14/61 processed
Row group 15/61 processed
Row group 16/61 processed
Row group 17/61 processed
Row group 18/61 processed
Row group 19/61 processed
Row group 20/61 processed
Row group 21/61 processed
Row group 22/61 processed
Row group 23/61 processed
Row group 24/61 processed
Row group 25/61 processed
Row group 26/61 processed
Row group 27/61 processed
Row group 28/61 processed
Row group 29/61 processed
Row group 30/61 processed
Row group 31/61 processed
Row group 32/61 processed
Row group 33/61 processed
Row group 34/61 processed
Row group 35/61 processed
Row group 36/61 processed
Row group 37/61 processed
Row group 38/61 processed
Row group 39/61 proce

In [27]:
import pyarrow.parquet as pq

path = "preprocessed_train.parquet"

pf = pq.ParquetFile(path)

print("====================================")
print("FINAL PREPROCESSING SANITY CHECK")
print("====================================")

print("\nRows:")
print(f"{pf.metadata.num_rows:,}")

print("\nColumns:")
print(pf.schema.names)

print("\nNumber of columns:")
print(len(pf.schema.names))

# Read only one row group for detailed checks
sample = pf.read_row_group(0).to_pandas()

print("\n------------------------------------")
print("Missing Values - Sample Row Group")
print("------------------------------------")

print(
    sample.isna().sum()[
        sample.isna().sum() > 0
    ]
)

print("\n------------------------------------")
print("Sales Statistics - Sample")
print("------------------------------------")

print(sample["sales"].describe())

print("\n------------------------------------")
print("Price Availability - Sample")
print("------------------------------------")

print(
    sample["price_available"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n------------------------------------")
print("Price Statistics - Available Prices")
print("------------------------------------")

print(
    sample.loc[
        sample["price_available"] == 1,
        "sell_price"
    ].describe()
)

print("\n------------------------------------")
print("Final Checks")
print("------------------------------------")

print(
    "Correct row count:",
    pf.metadata.num_rows == 58_327_370
)

print(
    "Correct column count:",
    len(pf.schema.names) == 23
)

print(
    "price_key removed:",
    "price_key" not in pf.schema.names
)

print(
    "price_available present:",
    "price_available" in pf.schema.names
)

print(
    "sales missing in sample:",
    sample["sales"].isna().sum()
)

print(
    "sell_price missing allowed:",
    sample["sell_price"].isna().sum() >= 0
)

FINAL PREPROCESSING SANITY CHECK

Rows:
58,327,370

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available']

Number of columns:
23

------------------------------------
Missing Values - Sample Row Group
------------------------------------
event_name_1     964576
event_type_1     964576
event_name_2    1046576
event_type_2    1046576
sell_price       367871
dtype: int64

------------------------------------
Sales Statistics - Sample
------------------------------------
count    1.048576e+06
mean     7.903070e-01
std      2.521917e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.940000e+02
Name: sales, dtype: float64

------------------------------------
Price Availability - Sample
------------------------------------
pri

## 16. Key Preprocessing Findings

The preprocessing pipeline successfully transformed the raw retail sales data into a time-series-friendly dataset.

### Key Findings

- The original sales data contained 58,327,370 daily product-store observations.
- The wide-format sales data was converted into long format.
- Calendar information was successfully merged using the daily identifier (`d`).
- Weekly selling prices were merged using `item_id`, `store_id`, and `wm_yr_wk`.
- The price join did not introduce duplicate rows or change the total number of observations.
- Sales contain no missing values, and zero-sales observations were preserved as valid demand observations.
- Approximately 21.09% of observations have missing selling prices.
- Previous analysis showed that observations with missing prices have zero sales, indicating that missing prices are associated with periods when the product-store combination was not actively selling.
- Missing selling prices were therefore not replaced with zero, mean, or median values.
- A `price_available` indicator was retained to explicitly represent price availability.
- Event-related missing values were preserved because they represent days without events rather than generic missing data.

## 17. Preprocessing Output

The final preprocessed dataset contains:

- Product information
- Store information
- Department and category information
- Daily sales
- Calendar features
- Event information
- SNAP indicators
- Selling price
- Price availability indicator

The final dataset contains **58,327,370 rows and 23 columns** and is stored in Parquet format as:

`preprocessed_train.parquet`

## 18. Conclusion

The raw retail sales data has been successfully cleaned, reshaped, and integrated with calendar and price information.

The resulting dataset is ready for the next stage: **feature engineering for multivariate time-series forecasting**.